# שיעור 18: אבטחת סוכני AI עם קבלות קריפטוגרפיות

## פנקס עבודות מעשי

פנקס זה עובר דרך ארבעה תרגילים:

1. **חתום על הקבלה הראשונה שלך** עבור קריאת כלי סוכן ואמת אותה.
2. **שנה את הקבלה** וצפה באימות שנכשל.
3. **בנה שרשרת שלוש קבלות** ואשר את שלמות השרשרת.
4. **עטוף קריאת כלי ממסגרת Microsoft Agent** כך שכל פעולה תפיק קבלה.

כל הפרימיטיבים הקריפטוגרפיים מיובאים מספריות מטופחות היטב (`pynacl` עבור Ed25519, `jcs` עבור JSON קנוני לפי RFC 8785, `hashlib` מספריית פייתון הסטנדרטית עבור SHA-256). הלוגיקה של הקבלה עצמה היא פייתון פשוט שניתן לקרוא לו ולערוך אותו.

הפעל את התאים לפי הסדר. כל חלק קצר ועצמאי.


## התקנה

התקן את שתי התלויות. לשתיהן רשיונות פתוחים (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## עזרי עזר

שני העזרים האלה מטפלים בקידוד base64url (ללא מילוי) ו-hashing SHA-256 של אובייקטים ארביטרריים. הם משאירים את שאר המחברת ממוקדת בלוגיקת הקבלה עצמה.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: חתום על הקבלה הראשונה שלך

דמיין שהסוכן שלנו עבור **Contoso Travel** זה עתה חיפש טיסות מסידני ללוס אנג'לס עבור לקוח. אנו רוצים לתעד את הקריאה לכלי זה כהוכחה חתומה כדי שמבקר עתידי יוכל לאמת זאת מבלי להסתמך עלינו.

### שלב 1.1: צור מפתח חתימה

בפרודקשן, מפתח החתימה של הסוכן היה נשמר במודול אבטחת חומרה (HSM), Azure Key Vault, או מאגר מוגן דומה. לשיעור זה אנו מייצרים מפתח חדש בזיכרון.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### שלב 1.2: בניית המטען של הקבלה

המטען מכיל את כל מה שאנחנו רוצים שהקבלה תאשר: מי פעל, באיזה כלי, עם אילו ארגומנטים, מה חזר, תחת איזו מדיניות, ומתי. אנחנו מבצעים hashing של הארגומנטים והתוצאה במקום לכלול אותם באופן ישיר כדי שהקבלה לא תחשוף תוכן רגיש.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### שלב 1.3: חתום והרכיב את הקבלה

שלושה שלבים:

1. בצע קנוניזציה של המטען באמצעות JCS כך ששני מימושים שמפיקים את אותה הקבלה הלוגית יפיקו ביטים זהים בדיוק.
2. חשב את גיבוב הביטים המקוננים באמצעות SHA-256.
3. חתום על הגיבוב באמצעות מפתח פרטי Ed25519.

החתימה מצורפת אז למטען המקורי ליצירת הקבלה הסופית.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### שלב 1.4: אימות הקבלה

האימות הופך את התהליך. אנו מסירים את החתימה, מחשבים מחדש את הקוד הסטנדרטי, ובודקים את החתימה מול המפתח הציבורי שבקבלה.

מבקר המבצע אימות זה אינו זקוק מאיתנו לשום דבר מלבד הקבלה עצמה. אין צורך לקרוא לשירות, אין צורך לשאול בספריית מפתחות, ואין צורך באמון.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

אתה אמור לראות `קבלה תקפה: True`. הסוכן הפיק את רשומת הביקורת הראשונה שלו עם חתימה קריפטוגרפית.


## Section 2: לשבש את הקבלה

כל המטרה של הקבלות היא שהן יהיו בעלות זיהוי לשיבוש. בואו נוכיח זאת.

נשנה בדיוק תו אחד בקבלה ונצפה באימות נכשל.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### מה בדיוק קרה?

כאשר שינינו את `policy_id`, הבייטים הקנוניים השתנו. ה-hash SHA-256 של אותם הבייטים השתנה. החתימה (שהייתה על ה-hash המקורי) כבר לא תואמת ל-hash החדש. האמתור מחזיר כהלכה `False`.

אין דרך לשנות אף שדה של הקבלה ועדיין לאשר אותה, אלא אם לתוקף יש את המפתח הפרטי. כל עוד המפתח הפרטי נמצא בארון המפתחות והמפתח הציבורי פורסם, שיבוש לא ניתן להסתיר.

נסו בעצמכם: שנו את `tool_name` או `agent_id` או `timestamp` בתא למעלה והריצו מחדש. כל שינוי מייצר קבלה לא תקפה.


## סעיף 3: לקשור קבלות אחת לשנייה

קבלה בודדת מגנה על פעולה אחת. רוב הסוכנים מבצעים פעולות רבות. כדי להפוך את כל הרצף לברור לשיבוש, אנו מקשרים כל קבלה לזו הקודמת על ידי הכללת ה־hash של הקבלה הקודמת בתוכן הקבלה החדשה.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

אם מישהו מסיר או משנה את סדר הקבלות, השרשרת נשברת בדיוק בנקודה הזו. האימות של כל קבלה מאוחרת יותר נכשל כי ה-`previous_receipt_hash` שלה כבר אינו תואם ל־hash בפועל של הקודמת לה.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

כעת שבר את השרשרת על ידי זיוף הקבלה האמצעית ובצע אימות מחדש. הקבלה המזויפת נכשלת בבדיקת החתימה שלה, והקבלה הבאה נכשלת בבדיקת קישור השרשרת שלה (כי `previous_receipt_hash` שלה כבר אינו תואם ל-hash של הקבלה האמצעית ששונתה).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

קבלה 0 עדיין מאומתת (לא שונתה ואין לה קבלה קודמת שתלויה בה). קבלה 1 נכשלת בבדיקת החתימה שלה כי שינינו את `tool_args_hash`. קבלה 2 נכשלת בבדיקת קישור השרשרת שלה כי `previous_receipt_hash` שלה חושב מול הקבלה המקורית (כעת שונתה) 1.

גם אם התוקף יחזור ויחתום מחדש על הקבלה ששונתה 1 (שהוא לא יכול לעשות ללא המפתח הפרטי), אי ההתאמה בקישור השרשרת בקבלה 2 עדיין תחשוף את ההתערבות. כדי להסתיר את השינוי, התוקף יצטרך לחתום מחדש על כל הקבלות מהנקודה שבה נעשה השינוי ואילך, דבר הדורש בעלות על המפתח הפרטי.


## Section 4: עטיפת קריאת כלי סוכן עם חתימת קבלה

בהטמעה אמיתית, אינך רוצה שכל יוצר סוכן יזכור לקרוא ל-`make_receipt`. אתה רוצה שחתימת הקבלה תהיה אוטומטית עבור כל קריאת כלי.

הנה התבנית הפשוטה ביותר: מחלקת עטיפה שלוקחת כל פונקציית כלי הניתנת לקריאה ומחזירה גרסה שמשחררת קבלה. זה מתאים לכל מסגרת סוכן, כולל Microsoft Agent Framework (`agent_framework.azure`).

אם אין לך פרויקט Azure AI Foundry מוגדר, הדוגמה המקומית למטה עדיין מדגימה את התבנית.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### אינטגרציה עם מסגרת הסוכנים של מיקרוסופט

העטיפה `ReceiptedTool` שלמעלה אינה תלויה במסגרת. כדי להשתמש בה בתוך סוכן שנבנה עם מסגרת הסוכנים של מיקרוסופט, יש לרשום את הפונקציה העטופה ככלי. סקיצה (אתה תחליף את המוק בדבר אמיתי לרישום כלי Azure AI Foundry):

```python
# קוד פסאודו המציג את צורת האינטגרציה.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     הוראות="אתה סוכן טיולים של Contoso ...",
#     tools=[receipted_lookup],   # הכלי עטוף, לא הפונקציה הגולמית
# )
# response = agent.run("מצא טיסות מסידני ללוס אנג'לס ביוני.")
#
# # לאחר הריצה, לכל קריאה לכלי שביצע הסוכן יש קבלה חתומה:
# audit_chain = receipted_lookup.receipts
```

מסגרת הסוכן אינה צריכה לדעת דבר על קבלות. חתימת הקבלה עטופה סביב הכלי, ולא מוטמעת בתוך המסגרת. כך מוסיפים מקוריות לקוד סוכן קיים מבלי לכתוב את הסוכן מחדש.


## סיכום ואתגר מתיחה

יש לך:

- יצרת זוג מפתחות Ed25519.
- בנית וחתמת קבלה לקריאת כלי סוכן.
- אימתת את הקבלה אופליין באמצעות המפתח הציבורי בלבד.
- זייפת קבלה וצפית בכישלון האימות.
- בנית רצף שלוש קבלות בשרשרת גיבוב.
- זייפת את האמצע של השרשרת וצפית גם בכישלון החתימה וגם בכישלון קישור השרשרת.
- עטפת פונקצית כלי בחתימת קבלה אוטומטית.

**אתגר מתיחה.** הרחב את סכמת הקבלה עם שדה `request_id` (UUID למעקב מבוזר). עדכן את `make_receipt` לכלול זאת, ואשר שקבלות עדיין מאומתות מקצה לקצה. לאחר מכן שנה את השדה לאחר החתימה ואשר שהאימות נכשל. זה מאלץ אותך להבין כיצד כל בית בהצפנה הקנונית תורם לחתימה.

**גבול חשוב.** קבלות מוכיחות שלושה דברים ורק שלושה דברים: שיוך (מפתח זה חתם על התוכן הזה), שלמות (התוכן לא השתנה מאז החתימה), וסדר (קבלה זו הגיעה אחרי קבלה אחרת). הן אינן מוכיחות כי פעולת הסוכן הייתה נכונה, שהמדיניות בשם `policy_id` הוערכה בפועל, או שהסוכן עקב אחרי כל חוק. הקבלות הן יסוד. הממשל הוא המערכת שאתה בונה מעליו.

קרא את קובץ ה-README של השיעור שוב כשהגבול הזה בראש. הטעות השכיחה ביותר שצוותים עושים עם קבלות היא להניח ש"יש לנו קבלות" פירושו "אנחנו נשלטים." זה לא נכון. קבלות עושות את התנהגות הסוכן ביקורתית. הן אינן עושות אותה נכונה.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
